# 01 — Architecture Walkthrough

This notebook walks through every component of Mythos from first principles:
RMSNorm → RoPE → Grouped Query Attention → SwiGLU → full Transformer

Goal: understand *why* each design decision was made, not just *what* it does.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import SVG, display

# Load Mythos components
from mythos.core.transformer import Mythos, ModelConfig
from mythos.core.attention import Attention
from mythos.core.norms import RMSNorm
from mythos.core.rope import build_rope_cache, apply_rope
from mythos.core.mlp import FeedForward

torch.manual_seed(42)
print('PyTorch:', torch.__version__)

## Architecture Diagram

In [ ]:
display(SVG('../assets/arch_diagram.svg'))

## 1. RMSNorm vs LayerNorm

LayerNorm: normalize by mean and std. RMSNorm: normalize by RMS only (no mean subtraction).

**Why RMSNorm?** ~7% fewer ops. Empirically no quality loss at this scale.

In [ ]:
d = 768
x = torch.randn(2, 16, d)

ln  = nn.LayerNorm(d)
rms = RMSNorm(d)

# LayerNorm: subtract mean, divide by std+eps, then affine
# RMSNorm: divide by RMS only
y_ln  = ln(x)
y_rms = rms(x)

print(f'LayerNorm  output mean: {y_ln.mean().item():+.4f}  std: {y_ln.std().item():.4f}')
print(f'RMSNorm    output mean: {y_rms.mean().item():+.4f}  std: {y_rms.std().item():.4f}')
print()
print(f'LayerNorm  params: {sum(p.numel() for p in ln.parameters())}')
print(f'RMSNorm    params: {sum(p.numel() for p in rms.parameters())}')

## 2. Rotary Position Embeddings (RoPE)

RoPE encodes position by rotating q and k vectors in pairs of dimensions.
The rotation angle for dimension pair i at position m is: m × θ^(-2i/d)

**Key property**: the dot product ⟨q_m, k_n⟩ depends only on (m-n) — pure relative position.

In [ ]:
seq_len, head_dim = 64, 64
cos, sin = build_rope_cache(seq_len, head_dim, theta=10000.0)
print(f'RoPE cache shape: cos={cos.shape}, sin={sin.shape}')

# Visualize rotation frequencies across dimensions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f172a')
for ax in axes:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values():
        spine.set_edgecolor('#334155')

# cos values heatmap
im = axes[0].imshow(cos.numpy().T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
axes[0].set_title('RoPE cos cache  (seq_len × head_dim/2)', color='#e2e8f0', fontsize=11)
axes[0].set_xlabel('position', color='#94a3b8')
axes[0].set_ylabel('dimension pair', color='#94a3b8')
plt.colorbar(im, ax=axes[0])

# Frequency spectrum
freqs = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2).float() / head_dim))
axes[1].plot(freqs.numpy(), color='#818cf8', linewidth=2)
axes[1].fill_between(range(len(freqs)), freqs.numpy(), alpha=0.3, color='#818cf8')
axes[1].set_title('RoPE frequencies per dimension pair', color='#e2e8f0', fontsize=11)
axes[1].set_xlabel('dimension pair index', color='#94a3b8')
axes[1].set_ylabel('frequency', color='#94a3b8')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('../assets/rope_frequencies.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 3. Grouped Query Attention (GQA)

MHA: n_heads Q heads, n_heads KV heads → full memory cost
GQA: n_heads Q heads, n_kv_heads KV heads → n_rep = n_heads/n_kv_heads

**Why does it work?** KV heads store context — sharing across query groups loses little information since adjacent query heads attend to similar patterns.

In [ ]:
# Compare MHA vs GQA parameter counts for 500M model
d_model = 1024
n_heads = 16

configs = [
    ('MHA (n_kv=16)', 16),
    ('GQA (n_kv=8)',  8),
    ('GQA (n_kv=4)',  4),
    ('MQA (n_kv=1)',  1),
]

head_dim = d_model // n_heads
print(f'{'Variant':<20} {'Wq+Wk+Wv params':>18} {'KV cache/token':>16} {'Rel. KV size':>14}')
print('-' * 72)
for name, n_kv in configs:
    wq_params = d_model * n_heads * head_dim
    wkv_params = 2 * d_model * n_kv * head_dim
    kv_per_token = 2 * n_kv * head_dim  # floats
    rel = n_kv / 16
    print(f'{name:<20} {wq_params + wkv_params:>18,} {kv_per_token:>16,} {rel:>14.0%}')

## 4. SwiGLU Feed-Forward

Classic FFN: `x → Linear → GeLU → Linear`

SwiGLU FFN: `x → (W1(x) ⊙ SiLU(W3(x))) → W2`

The gating mechanism lets the network learn which information to pass through.

In [ ]:
# Visualize SiLU (Swish) vs GeLU
x_plot = torch.linspace(-4, 4, 200)
silu = F.silu(x_plot)
gelu = F.gelu(x_plot)
relu = F.relu(x_plot)

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values():
    spine.set_edgecolor('#334155')

ax.plot(x_plot, silu.numpy(), color='#34d399', linewidth=2.5, label='SiLU (in SwiGLU)')
ax.plot(x_plot, gelu.numpy(), color='#818cf8', linewidth=2.5, label='GeLU', linestyle='--')
ax.plot(x_plot, relu.numpy(), color='#f59e0b', linewidth=2, label='ReLU', linestyle=':')
ax.axhline(0, color='#334155', linewidth=0.5)
ax.axvline(0, color='#334155', linewidth=0.5)
ax.set_title('Activation Functions', color='#e2e8f0', fontsize=12)
ax.set_xlabel('x', color='#94a3b8')
ax.set_ylabel('activation(x)', color='#94a3b8')
legend = ax.legend(facecolor='#0f172a', edgecolor='#334155', labelcolor='#e2e8f0')

plt.tight_layout()
plt.savefig('../assets/activations.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 5. Full Model: Parameter Count Breakdown

In [ ]:
from mythos.utils.config import load_config

# Instantiate 500M model
config_500m = ModelConfig(
    vocab_size=32000, d_model=1024, n_layers=40, n_heads=16, n_kv_heads=8,
    d_ff=4096, max_seq_len=2048
)
model = Mythos(config_500m)

# Component breakdown
def count_params(m):
    return sum(p.numel() for p in m.parameters())

total = model.get_num_params()
emb   = count_params(model.embedding)

attn_total = sum(count_params(l.attention) for l in model.layers)
ffn_total  = sum(count_params(l.feed_forward) for l in model.layers)
norm_total = sum(count_params(l.attn_norm) + count_params(l.ffn_norm) for l in model.layers)

print(f'Total parameters:     {total:>12,}  ({total/1e6:.1f}M)')
print(f'  Embedding (tied):   {emb:>12,}  ({100*emb/total:.1f}%)')
print(f'  Attention × {config_500m.n_layers}:    {attn_total:>12,}  ({100*attn_total/total:.1f}%)')
print(f'  FFN × {config_500m.n_layers}:           {ffn_total:>12,}  ({100*ffn_total/total:.1f}%)')
print(f'  Norms × {config_500m.n_layers}:          {norm_total:>12,}  ({100*norm_total/total:.1f}%)')

# Pie chart
fig, ax = plt.subplots(figsize=(6, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

sizes  = [emb, attn_total, ffn_total, norm_total]
labels = ['Embedding\n(tied)', f'Attention\n×{config_500m.n_layers}', f'FFN\n×{config_500m.n_layers}', 'Norms']
colors = ['#3b82f6', '#7c3aed', '#059669', '#d97706']
explode = (0.02, 0.02, 0.02, 0.05)

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct='%1.1f%%', startangle=90,
    textprops={'color': '#e2e8f0', 'fontsize': 10},
    pctdistance=0.8
)
for at in autotexts:
    at.set_color('#0f172a')
    at.set_fontweight('bold')

ax.set_title(f'Parameter Distribution — Mythos-500M ({total/1e6:.0f}M params)',
             color='#e2e8f0', fontsize=11, pad=15)
plt.tight_layout()
plt.savefig('../assets/param_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()